# DriftWise: UNSW-NB15 data audit
Chronological sorting, port cleaning, and the finding that all attacks occur early in the timeline. **Needs** `data/raw/UNSW-NB15_1.csv` and `NUSW-NB15_features.csv` in your Drive (they were not found in the Drive you used last, so upload them again first).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Check the uploaded files

In [ ]:
import os

data_path = "/content/drive/MyDrive/DriftWise/data/raw"

print("Files found in data/raw:")

for filename in os.listdir(data_path):
    print(filename)

Files found in data/raw:
UNSW-NB15_1.csv
NUSW-NB15_features.csv


Inspect the dataset

In [ ]:
import pandas as pd
import os

data_path = "/content/drive/MyDrive/DriftWise/data/raw"

file_path = os.path.join(data_path, "UNSW-NB15_1.csv")

df = pd.read_csv(file_path, header=None)

print("Dataset shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nNumber of columns:", len(df.columns))

/tmp/ipykernel_551/3495821613.py:8: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, header=None)


Dataset shape: (700001, 49)

First 5 rows:


,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0



Number of columns: 49


Identify the column names

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")

features_path = "/content/drive/MyDrive/DriftWise/data/raw/NUSW-NB15_features.csv"

features = pd.read_csv(features_path, encoding="cp1252")
print("Features file shape:", features.shape)
display(features.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Features file shape: (49, 4)


,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol
5,6,state,nominal,Indicates to the state and its dependent proto...
6,7,dur,Float,Record total duration
7,8,sbytes,Integer,Source to destination transaction bytes
8,9,dbytes,Integer,Destination to source transaction bytes
9,10,sttl,Integer,Source to destination time to live value


Assign the correct column names

In [ ]:
# Get the correct column names from the features file
column_names = features["Name"].str.strip().tolist()

# Assign the names to the dataset
df.columns = column_names

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows with proper column names:")
display(df.head())

Dataset shape: (700001, 49)

Column names:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']

First 5 rows with proper column names:


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,59.166.0.0,1390,149.171.126.6,53,udp,CON,0.001055,132,164,31,...,0,3,7,1,3,1,1,1,NaN,0
1,59.166.0.0,33661,149.171.126.9,1024,udp,CON,0.036133,528,304,31,...,0,2,4,2,3,1,1,2,NaN,0
2,59.166.0.6,1464,149.171.126.7,53,udp,CON,0.001119,146,178,31,...,0,12,8,1,2,2,1,1,NaN,0
3,59.166.0.5,3593,149.171.126.5,53,udp,CON,0.001209,132,164,31,...,0,6,9,1,1,1,1,1,NaN,0
4,59.166.0.3,49664,149.171.126.0,53,udp,CON,0.001169,146,178,31,...,0,7,9,1,1,1,1,1,NaN,0


Check attack labels and missing values

In [ ]:
# Clean extra spaces from column names
df.columns = df.columns.str.strip().str.replace(" ", "", regex=False)

# Check the target labels
print("Binary label distribution:")
print(df["Label"].value_counts(dropna=False))

print("\nAttack categories:")
print(df["attack_cat"].value_counts(dropna=False))

# Check missing values
print("\nColumns with missing values:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# Check data types
print("\nData types:")
print(df.dtypes.value_counts())

Binary label distribution:
Label
0    677786
1     22215
Name: count, dtype: int64

Attack categories:
attack_cat
NaN               677786
Generic             7522
Exploits            5409
 Fuzzers            5051
Reconnaissance      1759
DoS                 1167
Backdoors            534
Analysis             526
Shellcode            223
Worms                 24
Name: count, dtype: int64

Columns with missing values:
attack_cat    677786
dtype: int64

Data types:
int64      31
float64    10
object      8
Name: count, dtype: int64


Check the data types and duplicate records

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print("\nText/object columns:")
print(df.select_dtypes(include=["object"]).columns.tolist())

print("\nFirst 10 protocol values:")
print(df["proto"].value_counts().head(10))

print("\nFirst 10 state values:")
print(df["state"].value_counts().head(10))

Duplicate rows: 59210

Text/object columns:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'service', 'attack_cat']

First 10 protocol values:
proto
tcp       494737
udp       195479
arp         4062
ospf        2116
unas        1442
icmp         150
sctp         112
any           42
sep           34
mobile        28
Name: count, dtype: int64

First 10 state values:
state
FIN    487911
CON    187505
INT     21799
REQ      2429
CLO       111
URH       108
RST        74
ECO        26
ACC        22
PAR         4
Name: count, dtype: int64


Check whether the data has timestamps

In [ ]:
print("Start time data type:", df["Stime"].dtype)
print("Last time data type:", df["Ltime"].dtype)

print("\nFirst 10 start times:")
print(df["Stime"].head(10).to_list())

print("\nStart time summary:")
print(df["Stime"].describe())

print("\nAre start times already sorted?")
print(df["Stime"].is_monotonic_increasing)

Start time data type: int64
Last time data type: int64

First 10 start times:
[1421927414, 1421927414, 1421927414, 1421927414, 1421927414, 1421927414, 1421927414, 1421927415, 1421927415, 1421927415]

Start time summary:
count    7.000010e+05
mean     1.421941e+09
std      7.914429e+03
min      1.421927e+09
25%      1.421934e+09
50%      1.421942e+09
75%      1.421947e+09
max      1.421956e+09
Name: Stime, dtype: float64

Are start times already sorted?
False


Check the timestamp ordering

In [ ]:
print("Unique start timestamps:", df["Stime"].nunique())

print(
    "Rows out of order:",
    (df["Stime"].diff().dropna() < 0).sum()
)

print("\nRecords per timestamp (top 10):")
print(df["Stime"].value_counts().head(10))

print("\nEarliest timestamp:", df["Stime"].min())
print("Latest timestamp:", df["Stime"].max())

Unique start timestamps: 27568
Rows out of order: 107672

Records per timestamp (top 10):
Stime
1421927443    426
1421945101    226
1421945117    210
1421927444    209
1421945200    200
1421945151    169
1421945153    166
1421945230    165
1421945140    161
1421945221    160
Name: count, dtype: int64

Earliest timestamp: 1421927377
Latest timestamp: 1421955842


Create a chronological working copy

In [ ]:
# Create a separate chronological copy
df_sorted = df.sort_values(
    by="Stime",
    kind="stable"
).reset_index(drop=True)

print("Original dataset shape:", df.shape)
print("Sorted dataset shape:", df_sorted.shape)

print(
    "Chronologically sorted:",
    df_sorted["Stime"].is_monotonic_increasing
)

print("\nFirst 5 timestamps:")
print(df_sorted["Stime"].head().to_list())

print("\nLast 5 timestamps:")
print(df_sorted["Stime"].tail().to_list())

Original dataset shape: (700001, 49)
Sorted dataset shape: (700001, 49)
Chronologically sorted: True

First 5 timestamps:
[1421927377, 1421927377, 1421927377, 1421927377, 1421927381]

Last 5 timestamps:
[1421955842, 1421955842, 1421955842, 1421955842, 1421955842]


Save the chronological dataset

In [ ]:
import os

processed_path = "/content/drive/MyDrive/DriftWise/data/processed"

os.makedirs(processed_path, exist_ok=True)

output_file = os.path.join(
    processed_path,
    "UNSW_NB15_1_chronological.csv"
)

df_sorted.to_csv(output_file, index=False)

print("Dataset saved successfully!")
print("Location:", output_file)
print("Rows saved:", len(df_sorted))

Dataset saved successfully!
Location: /content/drive/MyDrive/DriftWise/data/processed/UNSW_NB15_1_chronological.csv
Rows saved: 700001


In [ ]:
import pandas as pd
import os

file_path = "/content/drive/MyDrive/DriftWise/data/processed/UNSW_NB15_1_chronological.csv"

print("File exists:", os.path.exists(file_path))
print("File size (MB):", round(os.path.getsize(file_path) / (1024 * 1024), 2))

df_check = pd.read_csv(file_path, nrows=5)

print("\nFirst 5 rows:")
display(df_check)

print("\nNumber of columns:", len(df_check.columns))
print("Column names:")
print(df_check.columns.tolist())

File exists: True
File size (MB): 162.88

First 5 rows:


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,10.40.182.1,0,224.0.0.5,0,ospf,INT,50.004337,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
1,10.40.85.1,0,224.0.0.5,0,ospf,INT,50.004341,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
2,10.40.182.1,0,224.0.0.5,0,ospf,INT,50.004337,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
3,10.40.85.1,0,224.0.0.5,0,ospf,INT,50.004341,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
4,192.168.241.243,259,192.168.241.243,49320,icmp,URH,0.000000,1780,0,64,...,0,2,2,2,2,1,1,2,NaN,0



Number of columns: 49
Column names:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label']


Check the target labels.

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/DriftWise/data/processed/UNSW_NB15_1_chronological.csv"

df = pd.read_csv(file_path)

print("Target label counts:")
print(df["Label"].value_counts().sort_index())

print("\nTarget label percentages:")
print((df["Label"].value_counts(normalize=True).sort_index() * 100).round(2))

print("\nMissing values in Label:")
print(df["Label"].isnull().sum())

/tmp/ipykernel_551/2587136401.py:5: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Target label counts:
Label
0    677786
1     22215
Name: count, dtype: int64

Target label percentages:
Label
0    96.83
1     3.17
Name: proportion, dtype: float64

Missing values in Label:
0


Check missing values and data types

In [ ]:
print("Missing values by column:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\nData types:")
print(df.dtypes.value_counts())

print("\nColumns stored as text:")
print(df.select_dtypes(include="object").columns.tolist())

Missing values by column:
attack_cat    677786
dtype: int64

Data types:
int64      31
float64    10
object      8
Name: count, dtype: int64

Columns stored as text:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'service', 'attack_cat']


Investigate duplicate records

In [ ]:
print("Total rows:", len(df))

print("Exact duplicate rows:", df.duplicated().sum())

print("\nDuplicate rows by target label:")
print(
    df.loc[df.duplicated(keep=False), "Label"]
      .value_counts()
      .sort_index()
)

print("\nFirst 5 duplicate rows:")
display(df[df.duplicated(keep=False)].head())

Total rows: 700001
Exact duplicate rows: 59211

Duplicate rows by target label:
Label
0    80485
1    12032
Name: count, dtype: int64

First 5 duplicate rows:


,srcip,sport,dstip,dsport,proto,state,dur,sbytes,dbytes,sttl,...,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,10.40.182.1,0,224.0.0.5,0,ospf,INT,50.004337,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
1,10.40.85.1,0,224.0.0.5,0,ospf,INT,50.004341,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
2,10.40.182.1,0,224.0.0.5,0,ospf,INT,50.004337,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
3,10.40.85.1,0,224.0.0.5,0,ospf,INT,50.004341,384,0,1,...,0,2,4,4,2,2,4,2,NaN,0
17,10.40.85.30,0,10.40.85.1,0,arp,CON,30.788803,138,84,0,...,0,2,2,2,2,2,2,2,NaN,0


Check duplicates across time

In [ ]:
# Count exact duplicate rows at each timestamp
duplicate_mask = df.duplicated(keep=False)

duplicate_rows = df.loc[duplicate_mask, ["Stime", "Label"]].copy()

print("Duplicate-group records by label:")
print(duplicate_rows["Label"].value_counts().sort_index())

print("\nEarliest timestamp among duplicate records:")
print(duplicate_rows["Stime"].min())

print("\nLatest timestamp among duplicate records:")
print(duplicate_rows["Stime"].max())

print("\nDuplicate records by time (first 10 timestamps):")
print(
    duplicate_rows.groupby("Stime")
    .size()
    .sort_index()
    .head(10)
)

Duplicate-group records by label:
Label
0    80485
1    12032
Name: count, dtype: int64

Earliest timestamp among duplicate records:
1421927377

Latest timestamp among duplicate records:
1421955825

Duplicate records by time (first 10 timestamps):
Stime
1421927377    4
1421927414    4
1421927415    4
1421927419    2
1421927421    2
1421927429    2
1421927434    2
1421927437    2
1421927443    2
1421927453    2
dtype: int64


Check suspicious values in the dataset

In [ ]:
import numpy as np

# Select numerical columns
numeric_df = df.select_dtypes(include="number")

# Check infinite values
infinite_counts = np.isinf(numeric_df).sum()
print("Columns containing infinite values:")
print(infinite_counts[infinite_counts > 0])

# Check missing values outside attack_cat
missing = df.drop(columns=["attack_cat"]).isnull().sum()
print("\nMissing values in other columns:")
print(missing[missing > 0])

# Check whether target labels are valid
print("\nUnique target labels:")
print(sorted(df["Label"].dropna().unique()))

Columns containing infinite values:
Series([], dtype: int64)

Missing values in other columns:
Series([], dtype: int64)

Unique target labels:
[np.int64(0), np.int64(1)]


Inspect categorical columns.

In [ ]:
categorical_cols = [
    "srcip", "sport", "dstip", "dsport",
    "proto", "state", "service"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Data type:", df[col].dtype)
    print("Unique values:", df[col].nunique(dropna=False))
    print("Missing values:", df[col].isnull().sum())
    print("Sample values:", df[col].drop_duplicates().head(10).tolist())


--- srcip ---
Data type: object
Unique values: 40
Missing values: 0
Sample values: ['10.40.182.1', '10.40.85.1', '192.168.241.243', '59.166.0.0', '59.166.0.6', '59.166.0.5', '59.166.0.3', '175.45.176.0', '10.40.85.30', '10.40.182.3']

--- sport ---
Data type: object
Unique values: 100263
Missing values: 0
Sample values: [0, 259, 49320, 1390, 33661, 1464, 3593, 49664, 32119, 2142]

--- dstip ---
Data type: object
Unique values: 44
Missing values: 0
Sample values: ['224.0.0.5', '192.168.241.243', '149.171.126.6', '149.171.126.9', '149.171.126.7', '149.171.126.5', '149.171.126.0', '149.171.126.4', '149.171.126.16', '149.171.126.2']

--- dsport ---
Data type: object
Unique values: 87815
Missing values: 0
Sample values: ['0', '49320', '0xc0a8', '53', '1024', '111', '80', '25', '52908', '41049']

--- proto ---
Data type: object
Unique values: 135
Missing values: 0
Sample values: ['ospf', 'icmp', 'udp', 'tcp', 'arp', 'igmp', 'sctp', 'udt', 'sep', 'sun-nd']

--- state ---
Data type: object
Un

Check port-number formats

In [ ]:
import pandas as pd

for col in ["sport", "dsport"]:
    values = df[col].astype(str).str.strip()

    is_decimal = values.str.fullmatch(r"\d+")
    is_hex = values.str.fullmatch(r"0[xX][0-9a-fA-F]+")

    print(f"\n--- {col} ---")
    print("Decimal values:", is_decimal.sum())
    print("Hexadecimal values:", is_hex.sum())
    print("Other formats:", (~(is_decimal | is_hex)).sum())

    print("\nExamples of other formats:")
    print(values[~(is_decimal | is_hex)].drop_duplicates().head(10).tolist())


--- sport ---
Decimal values: 699993
Hexadecimal values: 6
Other formats: 2

Examples of other formats:
['-']

--- dsport ---
Decimal values: 699938
Hexadecimal values: 56
Other formats: 7

Examples of other formats:
['-']


Clean the port columns

In [ ]:
df_clean = df.copy()

def convert_port(value):
    value = str(value).strip()

    if value == "-":
        return float("nan")

    try:
        return int(value, 0) if value.lower().startswith("0x") else int(value)
    except ValueError:
        return float("nan")

for col in ["sport", "dsport"]:
    df_clean[col] = df_clean[col].apply(convert_port)

print("Port columns cleaned.")

print("\nData types:")
print(df_clean[["sport", "dsport"]].dtypes)

print("\nMissing port values:")
print(df_clean[["sport", "dsport"]].isnull().sum())

print("\nSample values:")
print(df_clean[["sport", "dsport"]].head(10))

Port columns cleaned.

Data types:
sport     float64
dsport    float64
dtype: object

Missing port values:
sport     2
dsport    7
dtype: int64

Sample values:
     sport   dsport
0      0.0      0.0
1      0.0      0.0
2      0.0      0.0
3      0.0      0.0
4    259.0  49320.0
5  49320.0  49320.0
6   1390.0     53.0
7  33661.0   1024.0
8   1464.0     53.0
9   3593.0     53.0


Inspect numerical features

In [ ]:
# Numerical columns in the cleaned dataset
numeric_cols = df_clean.select_dtypes(include="number").columns.tolist()

# Exclude target and timestamp columns from this initial feature summary
exclude_cols = ["Label", "Stime", "Ltime"]
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

print("Number of numerical features:", len(feature_cols))

print("\nNumerical feature summary:")
display(df_clean[feature_cols].describe().T.round(2))

print("\nFeatures with zero variance:")
zero_variance = df_clean[feature_cols].nunique()
print(zero_variance[zero_variance <= 1])

Number of numerical features: 40

Numerical feature summary:


,count,mean,std,min,25%,50%,75%,max
sport,699999.0,3.221876e+04,1.933680e+04,0.0,15296.00,3.237800e+04,4.872700e+04,6.553500e+04
dsport,699994.0,1.541147e+04,9.112501e+05,0.0,53.00,1.968000e+03,2.498100e+04,5.389893e+08
dur,700001.0,8.600000e-01,2.594000e+01,0.0,0.00,3.000000e-02,3.500000e-01,8.786640e+03
sbytes,700001.0,5.009640e+03,4.489868e+04,0.0,424.00,2.024000e+03,3.806000e+03,1.367739e+07
dbytes,700001.0,4.833038e+04,1.860516e+05,0.0,304.00,3.380000e+03,2.243200e+04,1.465542e+07
sttl,700001.0,3.851000e+01,4.000000e+01,0.0,31.00,3.100000e+01,3.100000e+01,2.550000e+02
dttl,700001.0,3.198000e+01,3.045000e+01,0.0,29.00,2.900000e+01,2.900000e+01,2.540000e+02
sloss,700001.0,6.330000e+00,1.886000e+01,0.0,0.00,4.000000e+00,7.000000e+00,5.096000e+03
dloss,700001.0,2.133000e+01,6.473000e+01,0.0,0.00,5.000000e+00,1.600000e+01,5.483000e+03
Sload,700001.0,6.024271e+06,5.181492e+07,0.0,78859.15,5.362718e+05,1.078489e+06,5.268000e+09



Features with zero variance:
Series([], dtype: int64)


Verify the port conversion

In [ ]:
# Check the largest values in both port columns
print("Largest sport values:")
print(df_clean["sport"].nlargest(10).tolist())

print("\nLargest dsport values:")
print(df_clean["dsport"].nlargest(10).tolist())

# Show the original hexadecimal destination-port values
original_values = df["dsport"].astype(str).str.strip()
hex_mask = original_values.str.fullmatch(r"0[xX][0-9a-fA-F]+")

print("\nOriginal hexadecimal dsport values:")
print(original_values[hex_mask].drop_duplicates().tolist())

Largest sport values:
[65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0]

Largest dsport values:
[538989345.0, 538989345.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65535.0, 65534.0]

Original hexadecimal dsport values:
['0xc0a8', '0x20205321']


Correct the port conversion

In [ ]:
import numpy as np

def clean_port(value):
    value = str(value).strip()

    if value == "-":
        return np.nan

    try:
        if value.lower().startswith("0x"):
            number = int(value, 16)
        else:
            number = int(value)

        # Valid port range
        if 0 <= number <= 65535:
            return number

        return np.nan

    except ValueError:
        return np.nan


# Reconvert from the original dataframe, not df_clean
for col in ["sport", "dsport"]:
    df_clean[col] = df[col].apply(clean_port)

print("Largest sport:", df_clean["sport"].max())
print("Largest dsport:", df_clean["dsport"].max())

print("\nMissing port values:")
print(df_clean[["sport", "dsport"]].isnull().sum())

print("\nRows preserved:", len(df_clean))

Largest sport: 65535.0
Largest dsport: 65535.0

Missing port values:
sport     2
dsport    9
dtype: int64

Rows preserved: 700001


Check the target and feature columns

In [ ]:
# Target column: what the model must predict
target_col = "Label"

# Columns excluded from the first baseline
exclude_cols = [
    "Label",       # Target, not an input
    "attack_cat",  # Attack category; avoid target leakage
    "srcip",       # Source IP address
    "dstip",       # Destination IP address
    "Stime",       # Timestamp
    "Ltime"        # Timestamp
]

# Select all remaining input features
feature_cols = [
    col for col in df_clean.columns
    if col not in exclude_cols
]

print("Number of input features:", len(feature_cols))
print("\nInput features:")
print(feature_cols)

print("\nTarget distribution:")
print(df_clean[target_col].value_counts())

print("\nTarget missing values:", df_clean[target_col].isnull().sum())

Number of input features: 43

Input features:
['sport', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']

Target distribution:
Label
0    677786
1     22215
Name: count, dtype: int64

Target missing values: 0


Check categorical features

In [ ]:
# Identify categorical and numerical input features
X = df_clean[feature_cols]

categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_cols)
print("\nNumber of categorical features:", len(categorical_cols))

print("\nNumber of numerical features:", len(numerical_cols))

print("\nUnique values in categorical features:")
for col in categorical_cols:
    print(f"{col}: {X[col].nunique()} unique values")

Categorical features: ['proto', 'state', 'service']

Number of categorical features: 3

Number of numerical features: 40

Unique values in categorical features:
proto: 135 unique values
state: 16 unique values
service: 13 unique values


Prepare a chronological train/test split

In [ ]:
# Split chronologically: first 80% for training, last 20% for testing
split_index = int(len(df_clean) * 0.8)

train_df = df_clean.iloc[:split_index].copy()
test_df = df_clean.iloc[split_index:].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining time range:")
print(train_df["Stime"].min(), "to", train_df["Stime"].max())

print("\nTesting time range:")
print(test_df["Stime"].min(), "to", test_df["Stime"].max())

print("\nTraining label distribution:")
print(train_df["Label"].value_counts(normalize=True).mul(100).round(2))

print("\nTesting label distribution:")
print(test_df["Label"].value_counts(normalize=True).mul(100).round(2))

Training rows: 560000
Testing rows: 140001

Training time range:
1421927377 to 1421948806

Testing time range:
1421948806 to 1421955842

Training label distribution:
Label
0    96.03
1     3.97
Name: proportion, dtype: float64

Testing label distribution:
Label
0    100.0
Name: proportion, dtype: float64


Inspect attacks over time

In [ ]:
# Divide the chronological data into 10 time-ordered sections
df_clean["time_section"] = pd.qcut(
    df_clean["Stime"].rank(method="first"),
    q=10,
    labels=False
)

# Count normal and attack records in each section
time_summary = df_clean.groupby("time_section").agg(
    start_time=("Stime", "min"),
    end_time=("Stime", "max"),
    total_records=("Label", "size"),
    attack_records=("Label", "sum")
)

time_summary["attack_percent"] = (
    time_summary["attack_records"]
    / time_summary["total_records"] * 100
).round(2)

display(time_summary)

,start_time,end_time,total_records,attack_records,attack_percent
time_section,,,,,
0,1421927377,1421930202,70001,4948,7.07
1,1421930202,1421932604,70000,12510,17.87
2,1421932604,1421935434,70000,4757,6.80
3,1421935434,1421938628,70000,0,0.00
4,1421938628,1421941918,70000,0,0.00
5,1421941918,1421944901,70000,0,0.00
6,1421944901,1421945701,69999,0,0.00
7,1421945701,1421948806,70001,0,0.00
8,1421948806,1421951961,70000,0,0.00


Check the attack timeline more closely

In [ ]:
# Count attack categories across the 10 time sections
attack_summary = pd.crosstab(
    df_clean["time_section"],
    df_clean["attack_cat"].fillna("Normal")
)

display(attack_summary)

# Find the last timestamp containing an attack
attack_rows = df_clean[df_clean["Label"] == 1]

print("Total attack records:", len(attack_rows))
print("First attack timestamp:", attack_rows["Stime"].min())
print("Last attack timestamp:", attack_rows["Stime"].max())

# Count attacks before and after the original 80/20 split
cutoff_time = train_df["Stime"].max()

print("\nAttacks at or before training cutoff:",
      (attack_rows["Stime"] <= cutoff_time).sum())

print("Attacks after training cutoff:",
      (attack_rows["Stime"] > cutoff_time).sum())

attack_cat,Fuzzers,Analysis,Backdoors,DoS,Exploits,Generic,Normal,Reconnaissance,Shellcode,Worms
time_section,,,,,,,,,,
0,1769,0,24,298,1828,244,65053,688,89,8
1,2379,468,490,698,2463,5322,57490,598,79,13
2,903,58,20,171,1118,1956,65243,473,55,3
3,0,0,0,0,0,0,70000,0,0,0
4,0,0,0,0,0,0,70000,0,0,0
5,0,0,0,0,0,0,70000,0,0,0
6,0,0,0,0,0,0,69999,0,0,0
7,0,0,0,0,0,0,70001,0,0,0
8,0,0,0,0,0,0,70000,0,0,0


Total attack records: 22215
First attack timestamp: 1421927414
Last attack timestamp: 1421934448

Attacks at or before training cutoff: 22215
Attacks after training cutoff: 0


Find a suitable chronological evaluation period

In [ ]:
# Count records and attacks in consecutive time periods
attack_end_time = df_clean.loc[
    df_clean["Label"] == 1, "Stime"
].max()

before_attacks_end = df_clean[df_clean["Stime"] <= attack_end_time]
after_attacks_end = df_clean[df_clean["Stime"] > attack_end_time]

print("Last attack timestamp:", attack_end_time)

print("\nAt or before the last attack:")
print("Records:", len(before_attacks_end))
print("Attacks:", int(before_attacks_end["Label"].sum()))
print("Normal:", int((before_attacks_end["Label"] == 0).sum()))

print("\nAfter the last attack:")
print("Records:", len(after_attacks_end))
print("Attacks:", int(after_attacks_end["Label"].sum()))
print("Normal:", int((after_attacks_end["Label"] == 0).sum()))

print("\nUnique timestamps with attacks:",
      df_clean.loc[df_clean["Label"] == 1, "Stime"].nunique())

Last attack timestamp: 1421934448

At or before the last attack:
Records: 186812
Attacks: 22215
Normal: 164597

After the last attack:
Records: 513189
Attacks: 0
Normal: 513189

Unique timestamps with attacks: 6490


Save the dataset findings

In [ ]:
dataset_note = """
DriftWise - UNSW-NB15 Initial Data Audit

Total records: 700,001
Total attack records: 22,215
Total normal records: 677,786

Last attack timestamp: 1421934448
Records at or before last attack: 186,812
Records after last attack: 513,189
Attacks after last attack: 0

Finding:
The chronological dataset contains an attack-active period followed
by a long normal-only period. A simple chronological 80/20 split
therefore produces a test set with no attacks.

Research decision:
Use controlled synthetic streams with known drift for the primary
method validation. Evaluate on real network traffic separately,
reporting its temporal limitations. Do not randomly shuffle the
stream to manufacture evidence of temporal drift.
"""

print(dataset_note)